# Analiza Evenimentelor de Fotbal cu Apache Spark și TensorFlow

---

| | |
|---|---|
| **Student** | Măgureanu Stefan-Ionut |
| **Grupă** | 505 |
| **Materie** | Big Data |
| **Program de studii** | Master — Baze de Date și Tehnologii Software |
| **An de studiu** | Anul II |
| **An universitar** | 2025–2026 |
| **Dataset** | [Football Events — Kaggle](https://www.kaggle.com/datasets/secareanualin/football-events) |

## Cuprins

1. [Introducere](#1-introducere)
   - 1.1 [Prezentarea setului de date](#11-prezentarea-setului-de-date)
   - 1.2 [Obiectivele proiectului](#12-obiectivele-proiectului)
2. [Procesarea datelor cu Spark SQL și DataFrame API](#cerinta-2) *(Fișier: Cod_sursa_cerinta_2)*
3. [Metode ML cu Spark MLlib](#cerinta-3) *(Fișier: Cod_sursa_cerinta_3)*
4. [Data Pipeline](#cerinta-4) *(Fișier: Cod_sursa_cerinta_4)*
5. [UDF și Optimizarea Hiperparametrilor](#cerinta-5) *(Fișier: Cod_sursa_cerinta_5)*
6. [Deep Learning cu TensorFlow](#cerinta-6) *(Fișier: Cod_sursa_cerinta_6)*
7. [Spark Streaming](#cerinta-7) *(Fișier: Cod_sursa_cerinta_7)*

---
## 1. Introducere

### 1.1 Prezentarea setului de date

Setul de date utilizat este **Football Events**, disponibil pe platforma Kaggle la adresa:
> https://www.kaggle.com/datasets/secareanualin/football-events

Acesta conține date granulare despre evenimentele din meciuri de fotbal din **cinci ligi europene majore** (Bundesliga, Premier League, La Liga, Serie A, Ligue 1) pe parcursul **sezonelor 2011–2017**.

Setul este compus din **două fișiere CSV**:

#### `events.csv` — 941.009 înregistrări
Fiecare rând reprezintă un eveniment dintr-un meci:

| Coloană | Descriere |
|---|---|
| `id_odsp` | Identificatorul unic al meciului |
| `id_event` | Identificatorul unic al evenimentului |
| `time` | Minutul în care s-a produs evenimentul |
| `event_type` | Tipul evenimentului (0=Anunț, 1=Tentativă șut, 2=Corner, 3=Fault, 4=Cartonaș galben, etc.) |
| `event_team` | Echipa care a produs evenimentul |
| `player` | Jucătorul implicat |
| `is_goal` | 1 dacă evenimentul a dus la gol, 0 altfel |
| `location` | Zona de pe teren (1–19) |
| `bodypart` | Partea corpului folosită (1=picior drept, 2=picior stâng, 3=cap) |
| `situation` | Situația de joc (1=joc deschis, 2=fază fixă, 3=corner, 4=lovitură liberă) |
| `shot_place` | Locul în poartă unde a mers șutul (1–13) |
| `assist_method` | Metoda de assist (0=fără, 1=pasă, 2=centrare, 3=pasă cu capul, 4=pasă în adâncime) |
| `fast_break` | 1 dacă evenimentul a venit dintr-un contraatac rapid |

#### `ginf.csv` — 10.112 înregistrări
Fiecare rând reprezintă un meci:

| Coloană | Descriere |
|---|---|
| `id_odsp` | Identificatorul unic al meciului |
| `date` | Data meciului |
| `league` | Liga (D1=Bundesliga, E0=Premier League, SP1=La Liga, I1=Serie A, F1=Ligue 1) |
| `season` | Sezonul |
| `ht` / `at` | Echipa gazdă / oaspete |
| `fthg` / `ftag` | Goluri marcate acasă / în deplasare (Full Time) |
| `odd_h` / `odd_d` / `odd_a` | Cotele de pariuri pentru victorie gazdă / egalitate / victorie oaspete |

### 1.2 Obiectivele proiectului

Proiectul urmărește aplicarea tehnicilor de **Big Data** studiate la curs pe un dataset real de fotbal, cu scopul de a extrage insight-uri relevante și de a construi modele predictive. Obiectivele concrete sunt:

**O1 — Analiză exploratorie distribuită (Cerința 2)**  
Utilizarea Apache Spark (DataFrame API și Spark SQL) pentru curățarea, transformarea și analiza statistică a celor ~950.000 de evenimente, identificând pattern-uri în distribuția golurilor, eficiența șuturilor și performanța echipelor.

**O2 — Modele de Machine Learning cu Spark MLlib (Cerința 3)**  
- **Clasificare:** Predicția dacă o tentativă de șut va fi gol, folosind Random Forest Classifier pe baza caracteristicilor șutului (locație, parte corp, situație).
- **Regresie:** Predicția numărului total de goluri dintr-un meci pe baza statisticilor agregate (corneruri, fault-uri, șuturi pe poartă).

**O3 — Data Pipeline reproductibil (Cerința 4)**  
Construirea unui pipeline ETL complet care automatizează preprocesarea (encoding, normalizare, asamblare features) și poate fi aplicat consistent pe date noi.

**O4 — UDF și tuning avansat (Cerința 5)**  
Definirea unei funcții personalizate pentru calculul unui „scor de calitate al șutului” și optimizarea hiperparametrilor modelului prin Cross-Validation.

**O5 — Deep Learning cu TensorFlow (Cerința 6)**  
Construirea și antrenarea unei rețele neuronale feed-forward pentru clasificarea binară a șuturilor, cu compararea performanței față de modelul MLlib.

**O6 — Inferență în timp real (Cerința 7)**  
Simularea unui flux de date de la un meci live și aplicarea modelului antrenat pentru predicții în timp real cu Spark Structured Streaming.

### 1.3 Previzualizare rapidă a datelor

In [1]:
import os
# Notebook-urile stau în subfolderul notebooks/; ne asigurăm că lucrăm din rădăcina proiectului
# (unde se află data/, models/, plots/), astfel încât toate căile relative să funcționeze.
if not os.path.isdir('data') and os.path.isdir(os.path.join('..', 'data')):
    os.chdir('..')
# Directoarele de output trebuie să existe înainte de orice savefig()/save()
os.makedirs('plots', exist_ok=True)
os.makedirs('models', exist_ok=True)
print('Working directory:', os.getcwd())

Working directory: /Users/stefan/Documents/football-events-analysis


In [2]:
import pandas as pd

events_preview = pd.read_csv('data/events.csv', nrows=5)
print(f'Coloane events.csv ({len(events_preview.columns)}): {list(events_preview.columns)}')
events_preview

Coloane events.csv (22): ['id_odsp', 'id_event', 'sort_order', 'time', 'text', 'event_type', 'event_type2', 'side', 'event_team', 'opponent', 'player', 'player2', 'player_in', 'player_out', 'shot_place', 'shot_outcome', 'is_goal', 'location', 'bodypart', 'assist_method', 'situation', 'fast_break']


,id_odsp,id_event,sort_order,time,text,event_type,event_type2,side,event_team,opponent,...,player_in,player_out,shot_place,shot_outcome,is_goal,location,bodypart,assist_method,situation,fast_break
0,UFot0hit/,UFot0hit1,1,2,Attempt missed. Mladen Petric (Hamburg) left f...,1,12.0,2,Hamburg SV,Borussia Dortmund,...,NaN,NaN,6.0,2.0,0,9.0,2.0,1,1.0,0
1,UFot0hit/,UFot0hit2,2,4,"Corner, Borussia Dortmund. Conceded by Dennis...",2,NaN,1,Borussia Dortmund,Hamburg SV,...,NaN,NaN,NaN,NaN,0,NaN,NaN,0,NaN,0
2,UFot0hit/,UFot0hit3,3,4,"Corner, Borussia Dortmund. Conceded by Heiko ...",2,NaN,1,Borussia Dortmund,Hamburg SV,...,NaN,NaN,NaN,NaN,0,NaN,NaN,0,NaN,0
3,UFot0hit/,UFot0hit4,4,7,Foul by Sven Bender (Borussia Dortmund).,3,NaN,1,Borussia Dortmund,Hamburg SV,...,NaN,NaN,NaN,NaN,0,NaN,NaN,0,NaN,0
4,UFot0hit/,UFot0hit5,5,7,Gokhan Tore (Hamburg) wins a free kick in the ...,8,NaN,2,Hamburg SV,Borussia Dortmund,...,NaN,NaN,NaN,NaN,0,2.0,NaN,0,NaN,0


In [3]:
ginf_preview = pd.read_csv('data/ginf.csv', nrows=5)
print(f'Coloane ginf.csv ({len(ginf_preview.columns)}): {list(ginf_preview.columns)}')
ginf_preview

Coloane ginf.csv (18): ['id_odsp', 'link_odsp', 'adv_stats', 'date', 'league', 'season', 'country', 'ht', 'at', 'fthg', 'ftag', 'odd_h', 'odd_d', 'odd_a', 'odd_over', 'odd_under', 'odd_bts', 'odd_bts_n']


,id_odsp,link_odsp,adv_stats,date,league,season,country,ht,at,fthg,ftag,odd_h,odd_d,odd_a,odd_over,odd_under,odd_bts,odd_bts_n
0,UFot0hit/,/soccer/germany/bundesliga-2011-2012/dortmund-...,True,2011-08-05,D1,2012,germany,Borussia Dortmund,Hamburg SV,3,1,1.56,4.41,7.42,NaN,NaN,NaN,NaN
1,Aw5DflLH/,/soccer/germany/bundesliga-2011-2012/augsburg-...,True,2011-08-06,D1,2012,germany,FC Augsburg,SC Freiburg,2,2,2.36,3.60,3.40,NaN,NaN,NaN,NaN
2,bkjpaC6n/,/soccer/germany/bundesliga-2011-2012/werder-br...,True,2011-08-06,D1,2012,germany,Werder Bremen,Kaiserslautern,2,0,1.83,4.20,4.80,NaN,NaN,NaN,NaN
3,CzPV312a/,/soccer/france/ligue-1-2011-2012/paris-sg-lori...,True,2011-08-06,F1,2012,france,Paris Saint-Germain,Lorient,0,1,1.55,4.50,9.40,NaN,NaN,NaN,NaN
4,GUOdmtII/,/soccer/france/ligue-1-2011-2012/caen-valencie...,True,2011-08-06,F1,2012,france,Caen,Valenciennes,1,0,2.50,3.40,3.45,NaN,NaN,NaN,NaN


In [4]:
import subprocess

events_lines = int(subprocess.check_output(['wc', '-l', 'data/events.csv']).split()[0]) - 1
ginf_lines   = int(subprocess.check_output(['wc', '-l', 'data/ginf.csv']).split()[0]) - 1

print(f'events.csv : {events_lines:,} înregistrări')
print(f'ginf.csv   : {ginf_lines:,} meciuri')
print(f'Total      : {events_lines + ginf_lines:,} rânduri')

events.csv : 941,009 înregistrări
ginf.csv   : 10,112 meciuri
Total      : 951,121 rânduri


---
> **Notă de navigare:** Cerințele 2–7 sunt implementate în fișiere notebook separate, denumite conform convenției `505_Magureanu_Stefan_Ionut-Cod_sursa_cerinta_<n>.ipynb`. Fiecare notebook este independent și poate fi rulat separat după activarea venv-ului.
>
> ```bash
> source venv/bin/activate
> jupyter notebook
> ```